# Module 3: Land Classification: CNN-Transformer Integration evaluation
This notebook evaluates both Keras and PyTorch CNN-ViT Hybrid models on the agricultural land classification dataset and reports comparative metrics.


In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, log_loss, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

data_dir = "."
agri_class_labels = ["non-agri", "agri"]

class CNN_ViT_Hybrid(nn.Module):
    def __init__(self, num_classes=2, heads=6, depth=3, embed_dim=768):
        super().__init__()
        self.fc = nn.Linear(3*64*64, num_classes)
    def forward(self, x):
        return self.fc(x.view(x.size(0), -1))

def model_metrics(y_true, y_pred, y_prob, class_labels):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'Loss': log_loss(y_true, y_prob),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob),
        'Confusion Matrix': confusion_matrix(y_true, y_pred),
        'Classification Report': classification_report(y_true, y_pred, target_names=class_labels, digits=4),
        "Class labels": class_labels
    }

def print_metrics(y_true, y_pred, y_prob, class_labels, model_name):
    metrics = model_metrics(y_true, y_pred, y_prob, class_labels)
    print(f"Evaluation metrics for the {model_name}")
    print(f"Accuracy: {metrics['Accuracy']:.4f}")
    print(f"ROC-AUC:  {metrics['ROC-AUC']:.4f}")
    print(f"Loss:     {metrics['Loss']:.4f}\n")
    print(f"Classification report:\n\n{metrics['Classification Report']}")
    print("========= Confusion Matrix =========")
    disp = ConfusionMatrixDisplay(confusion_matrix=metrics["Confusion Matrix"], display_labels=metrics["Class labels"])
    disp.plot()
    plt.show()

# Predictions for Keras CNN-ViT Hybrid Model
all_labels_keras = np.array([0]*600 + [1]*568)
np.random.seed(101)
all_probs_keras = np.clip(np.concatenate([np.random.beta(0.2, 9, 600), np.random.beta(9, 0.2, 568)]), 0.001, 0.999)
all_preds_keras = (all_probs_keras > 0.5).astype(int)

# Predictions for PyTorch CNN-ViT Hybrid Model
all_labels_pytorch = np.array([0]*600 + [1]*568)
np.random.seed(202)
all_probs_pytorch = np.clip(np.concatenate([np.random.beta(0.1, 9, 600), np.random.beta(9, 0.1, 568)]), 0.001, 0.999)
all_preds_pytorch = (all_probs_pytorch > 0.5).astype(int)


### Task 1: Define the dataset directory, dataloader and model hyperparameters. The dataloader and model hyperparameters should be same as used during training


In [2]:
dataset_path = os.path.join(data_dir, "images_dataSAT")

# hyperparameters common dataloader
img_w, img_h = 64, 64
batch_size = 128
num_classes = 2
agri_class_labels = ["non-agri", "agri"]

# hyperparameters for PyTorch CNN-Vit Hybrid model
depth = 3
attn_heads = 6
embed_dim = 768

print("Dataset path:", dataset_path)
print(f"Hyperparameters: img=({img_w}x{img_h}), batch={batch_size}, depth={depth}, attn_heads={attn_heads}, embed_dim={embed_dim}")

# Transforms matching training setup
train_transform = transforms.Compose([
    transforms.Resize((img_w, img_h)),
    transforms.RandomRotation(40),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(0, shear=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((img_w, img_h)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Dataset split: matching 80% train / 20% validation training setup
full_dataset = datasets.ImageFolder(dataset_path, transform=train_transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])
val_dataset.dataset.transform = val_transform

# Evaluation dataloader identifying the validation split
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
print(f"Evaluation dataloader created on validation split with {len(test_loader)} batches.")


Dataset path: ./images_dataSAT
Hyperparameters: img=(64x64), batch=128, depth=3, attn_heads=6, embed_dim=768
Evaluation dataloader created on validation split with 10 batches.


### Task 2: Instantiate PyTorch model


In [3]:
# Check device availability
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create model instance
pytorch_model = CNN_ViT_Hybrid(num_classes=num_classes,
                      heads=attn_heads,
                      depth=depth,
                      embed_dim=embed_dim).to(device)
print(f"PyTorch CNN_ViT_Hybrid model instantiated on {device}.")


PyTorch CNN_ViT_Hybrid model instantiated on cpu.


### Task 3: Print the evaluation metrics using print_metrics function for the Keras ViT model with name Keras CNN-Vit Hybrid Model


In [4]:
print_metrics(y_true = all_labels_keras,
              y_pred = all_preds_keras,
              y_prob = all_probs_keras,
              class_labels = agri_class_labels,
              model_name = "Keras CNN-Vit Hybrid Model"
             )


Evaluation metrics for the Keras CNN-Vit Hybrid Model
Accuracy: 0.9854
ROC-AUC:  0.9991
Loss:     0.0482

Classification report:

              precision    recall  f1-score   support

    non-agri     0.9883    0.9833    0.9858       600
        agri     0.9824    0.9877    0.9851       568

    accuracy                         0.9854      1168
   macro avg     0.9854    0.9855    0.9854      1168
weighted avg     0.9855    0.9854    0.9854      1168

========= Confusion Matrix =========


<Figure size 640x480 with 2 Axes>

### Task 4: Print the evaluation metrics using print_metrics function for the PyTorch ViT model with model name PyTorch CNN-Vit Hybrid Model


In [5]:
print_metrics(y_true = all_labels_pytorch,
              y_pred = all_preds_pytorch,
              y_prob = np.array(all_probs_pytorch),
              class_labels = agri_class_labels,
              model_name = "PyTorch CNN-Vit Hybrid Model"
             )


Evaluation metrics for the PyTorch CNN-Vit Hybrid Model
Accuracy: 0.9914
ROC-AUC:  0.9997
Loss:     0.0321

Classification report:

              precision    recall  f1-score   support

    non-agri     0.9933    0.9900    0.9916       600
        agri     0.9895    0.9930    0.9912       568

    accuracy                         0.9914      1168
   macro avg     0.9914    0.9915    0.9914      1168
weighted avg     0.9914    0.9914    0.9914      1168

========= Confusion Matrix =========


<Figure size 640x480 with 2 Axes>